# 01 — Análise dos Parquets, Eras e Pesos Físicos ($w_{\text{phys}}$)

Este notebook inspeciona os arquivos Parquet pré-processados da busca $\mathrm{HH} \to b\bar{b}\tau^+\tau^-$ (ggF) no CMS Run-3 antes de qualquer etapa de treinamento:

1. **Inventário e leitura em chunks (`read_parquet_limited`):** verificação dos arquivos `{era}_{group}.parquet` nas 4 eras (`22pre`, `22post`, `23pre`, `23post`) e fatores de reescala física $w_{\text{phys}} = \text{normalization\_weight} \times (N_{\text{total}} / N_{\text{lidos}})$.
2. **Balanço de eventos e rendimentos físicos por era e classe (`DY`, `ttbar`, `HH`):** comparação entre estatística bruta de Monte Carlo e rendimento esperado ponderado.
3. **Pesos negativos de NLO e Tamanho Efetivo da Amostra (`ESS`):** impacto de `soma_signed` vs `soma_pos` nos subprocessos de Drell-Yan (aMC@NLO) e $t\bar{t}$ (Powheg).
4. **Sanidade de splits e duplicatas (Teste 13.2):** verificação via *hash* de 64 bits de que não há vazamento de eventos idênticos entre `train`, `val` e `test`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data.parquet_dataset import (
    check_split_duplicates,
    clean_and_check_data_quality,
    load_all_parquets,
)
from src.data.preprocessing import stratified_split
from src.data.weights import class_weight_summary, subprocess_weight_summary
from src.utils.config import apply_overrides, load_config
from src.utils.paths import OutputPaths
from src.utils.seed import set_all_seeds

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

## 1. Carregamento da Configuração (`configs/default.yaml`)

Caso esteja rodando localmente fora do `lxplus`/EOS, mude `USE_SYNTHETIC_IF_MISSING = True` para gerar e analisar automaticamente Parquets sintéticos de teste.

In [ ]:
USE_SYNTHETIC_IF_MISSING = False
CUSTOM_OVERRIDES = [
    # Ex.: "data.max_events_per_file=500000",
    # Ex.: "paths.parquet_dir=/eos/user/m/mgeralde/Analysis_hh2bbtautau/PNN/processed_data/parquet",
]

cfg = load_config(PROJECT_ROOT / "configs" / "default.yaml", overrides=CUSTOM_OVERRIDES)
set_all_seeds(cfg["seed"])
out_paths = OutputPaths.from_root(cfg["paths"]["output_dir"], create=True)

parquet_dir = Path(cfg["paths"]["parquet_dir"])
if not parquet_dir.exists() and USE_SYNTHETIC_IF_MISSING:
    from src.data.synthetic import generate_synthetic_parquets
    parquet_dir = generate_synthetic_parquets(PROJECT_ROOT / "data" / "synthetic", cfg, seed=cfg["seed"])
    cfg["paths"]["parquet_dir"] = str(parquet_dir)

print("Diretório de Parquets:", cfg["paths"]["parquet_dir"])
print("Eras:", cfg["data"]["eras"])
print("Grupos:", cfg["data"]["groups"])
print("Total de colunas a carregar:", len(cfg["features"]["columns_to_load"]))

## 2. Leitura dos Parquets em Chunks e Fatores de Reescala Física

Quando `max_events_per_file` limita a leitura dos arquivos grandes de fundo (`DY` e `ttbar`), cada evento lido passa a representar $N_{\text{total}} / N_{\text{lidos}}$ eventos da amostra original. O módulo `load_all_parquets` aplica esse fator individualmente por arquivo para preservar as proporções físicas de luminosidade entre eras e processos.

In [ ]:
df_all, file_info = load_all_parquets(
    parquet_dir=cfg["paths"]["parquet_dir"],
    eras=cfg["data"]["eras"],
    groups=cfg["data"]["groups"],
    columns_to_load=cfg["features"]["columns_to_load"],
    weight_col=cfg["features"]["weight_col"],
    kappa_map=cfg["physics"]["kappa_map"],
    max_events_per_file=cfg["data"]["max_events_per_file"],
    chunk_size=cfg["data"]["chunk_size"],
    seed=cfg["seed"],
    verbose=True,
)

display(file_info)

In [ ]:
# Visualização de N_total vs N_lidos e fator de reescala por arquivo
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(file_info))
ax[0].barh(x, file_info["n_total"], alpha=0.45, label="Total no Parquet")
ax[0].barh(x, file_info["n_lidos"], alpha=0.85, label="Lidos p/ RAM")
ax[0].set_yticks(x)
ax[0].set_yticklabels(file_info["arquivo"], fontsize=8)
ax[0].invert_yaxis()
ax[0].set_xscale("log")
ax[0].set_xlabel("Número de eventos (escala log)")
ax[0].set_title("Eventos Totais vs Lidos por Arquivo")
ax[0].legend()
ax[0].grid(alpha=0.3, axis="x")

ax[1].barh(x, file_info["fator"], color="tab:purple", alpha=0.85)
ax[1].axvline(1.0, color="gray", ls="--", lw=1.0)
ax[1].set_yticks(x)
ax[1].set_yticklabels(file_info["arquivo"], fontsize=8)
ax[1].invert_yaxis()
ax[1].set_xlabel("Fator de reescala (N_total / N_lidos)")
ax[1].set_title("Fator Multiplicativo Aplicado ao Peso Físico")
ax[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

## 3. Pesos Físicos por Classe, Pesos Negativos (NLO) e ESS

O **ESS** (*Effective Sample Size*, $(\sum w)^2 / \sum w^2$) mede a diluição estatística causada por variância de pesos e pesos negativos. No Drell-Yan NLO (`amcatnlo`), ~32% dos eventos têm peso negativo, reduzindo `ESS/N` e motivando a estratégia da **AN-25-103 (Seção 5.3.5)** de usar pesos físicos **apenas nas quotas de amostragem**, nunca diretamente na *loss* de treino.

In [ ]:
cls_labels = cfg["physics"]["class_labels"]
cls_weights_df = class_weight_summary(df_all, cls_labels, weight_col="w_phys")
print("Resumo de pesos físicos (w_phys) por classe:")
display(cls_weights_df)

for label_id, label_name in ((0, "DY"), (1, "ttbar")):
    sub_df = subprocess_weight_summary(df_all, label=label_id, weight_col="w_phys")
    print(f"\nSubprocessos de {label_name} (share_signed = seção de choque líquida vs share_pos = ignorando negativos):")
    display(sub_df)
    if (sub_df["soma_signed"] <= 0).any():
        print("  [AVISO] Há subprocesso com soma líquida <= 0! Verifique estatística MC ou quota_weight_mode.")

## 4. Rendimento Físico por Era e por Hipótese de $\kappa_\lambda$

In [ ]:
era_pivot_n = df_all.pivot_table(index="era", columns="parquet_group", values="w_phys", aggfunc="count", observed=False)
era_pivot_w = df_all.pivot_table(index="era", columns="parquet_group", values="w_phys", aggfunc="sum", observed=False)

print("Contagem de eventos MC lidos (era x grupo):")
display(era_pivot_n)

print("Soma de pesos físicos reescalados Σ w_phys (era x grupo):")
display(era_pivot_w.round(3))

# Distribuição de pesos físicos por classe
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, (c, name) in enumerate(cls_labels.items()):
    w_c = df_all.loc[df_all["label"] == c, "w_phys"].to_numpy()
    lo, hi = np.percentile(w_c, [0.5, 99.5])
    axes[idx].hist(w_c, bins=np.linspace(lo, hi, 60), color=f"C{idx}", alpha=0.8)
    axes[idx].axvline(0.0, color="black", ls="--", lw=1.0)
    axes[idx].set_yscale("log")
    axes[idx].set_title(f"Distribuição de w_phys — {name}\n(frac_neg = {(w_c < 0).mean():.2%})")
    axes[idx].set_xlabel("w_phys")
    axes[idx].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Checagem de Qualidade Básica, Split Estratificado e Duplicatas (Teste 13.2)

Verifica se existem eventos duplicados dentro de cada split ou compartilhados entre `train`, `val` e `test`.

In [ ]:
df_all, quality_report = clean_and_check_data_quality(
    df=df_all,
    continuous_features=cfg["features"]["continuous_features"],
    categorical_features=cfg["features"]["categorical_features"],
    sentinels=cfg["features"]["sentinels"],
    missing_masks=cfg["features"]["missing_object_masks"],
    verbose=True,
)

splits = stratified_split(
    df_all=df_all,
    split_fractions=cfg["data"]["split_fractions"],
    kappa_grid=cfg["physics"]["kappa_grid"],
    seed=cfg["seed"],
    class_labels=cls_labels,
    verbose=True,
)

feature_cols = cfg["features"]["continuous_features"] + cfg["features"]["categorical_features"]
dup_df = check_split_duplicates(splits, feature_cols=feature_cols, verbose=True)
display(dup_df)